In [ ]:
# Install the face-recognition library
!pip install -q insightface onnxruntime

In [ ]:
import sys; sys.path.append("..") 
from src import generate, evaluate, config
import importlib; importlib.reload(generate); importlib.reload(evaluate)

In [ ]:
# Generate a batch of 10 across different seeds, with the good prompt
generate.generate_for_eval(
    n=10,
    prompt="a photo of sks man, color photograph, sharp focus, natural daylight",
    negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
    steps=40, guidance_scale=7.0, base_seed=0,
)

In [ ]:
# Measure identity similarity + novelty
res = evaluate.evaluate(train_dir=config.TRAIN_DIR)
res

In [ ]:
generate.generate_for_eval(
    n=10,
    prompt="a photo of sks man",
    negative_prompt=None,
    steps=30, guidance_scale=7.5, base_seed=0,
)
res_plain = evaluate.evaluate(train_dir=config.TRAIN_DIR)
res_plain

In [ ]:
# Evaluate rank 16
generate.generate_for_eval(
    n=10,
    prompt="a photo of sks man, color photograph, sharp focus, natural daylight",
    negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
    steps=40, guidance_scale=7.0, base_seed=0,
)
evaluate.evaluate(train_dir=config.TRAIN_DIR)

Generate the baseline images

In [16]:
import torch, gc
from diffusers import StableDiffusionPipeline
from src import config

base_dir = config.LORA_DIR.parent / "generated_baseline"
base_dir.mkdir(parents=True, exist_ok=True)
for p in base_dir.glob("*.png"):
    p.unlink()

pipe = StableDiffusionPipeline.from_pretrained(config.BASE_MODEL).to(config.DEVICE)
pipe.safety_checker = None

for k in range(10):
    g = torch.Generator(device=config.DEVICE).manual_seed(k)
    img = pipe(
        "a photo of a man, color photograph, sharp focus, natural daylight",
        negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
        num_inference_steps=40, guidance_scale=7.0, generator=g,
    ).images[0]
    img.save(base_dir / f"base_{k:03d}.png")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("baseline images saved to", base_dir)

100%|██████████| 40/40 [01:08<00:00,  1.71s/it]


baseline images saved to /Users/bezawit/Courses/Programming_for_AI_ML_II/few-shot-face-gen/outputs/generated_baseline


Score them

In [17]:
from src import evaluate, config
evaluate.evaluate(generated_dir=base_dir, train_dir=config.TRAIN_DIR)

Generated images:         10
  with a detectable face: 8  (80%)
Real reference faces:     4
--------------------------------------------------------
      base_000.png:  identity=-0.056   nearest-train=0.013
      base_002.png:  identity=-0.006   nearest-train=0.089
      base_003.png:  identity=-0.005   nearest-train=0.058
      base_004.png:  identity=0.115   nearest-train=0.188
      base_005.png:  identity=0.008   nearest-train=0.037
      base_006.png:  identity=-0.006   nearest-train=0.027
      base_007.png:  identity=0.053   nearest-train=0.133
      base_009.png:  identity=0.007   nearest-train=0.068
--------------------------------------------------------
Mean identity similarity: 0.014
Best / worst:             0.115 / -0.056
Above threshold 0.35:     0/8 (0%)
Mean nearest-train sim:   0.077 (well below 1.0 = novel, not copied)


{'n_total': 10,
 'n_faces': 8,
 'mean_identity': 0.013638429343700409,
 'best_identity': 0.11507315933704376,
 'worst_identity': -0.05622757971286774,
 'pass_rate': 0.0,
 'threshold': 0.35,
 'similarities': [-0.05622757971286774,
  -0.005700967274606228,
  -0.005266804248094559,
  0.11507315933704376,
  0.007787753827869892,
  -0.006346451118588448,
  0.053274646401405334,
  0.006513677537441254],
 'mean_novelty': 0.07659863028675318}

A real-photo ceiling.


In [18]:
import numpy as np
from src import evaluate, config

test_embs = evaluate._embeddings_for_dir(config.TEST_DIR)
train_embs = evaluate._embeddings_for_dir(config.TRAIN_DIR)
train_mat = np.stack(list(train_embs.values()))

sims = []
for name, e in test_embs.items():
    mean_sim = float(np.mean(train_mat @ e))
    best_sim = float(np.max(train_mat @ e))
    sims.append(mean_sim)
    print(f"{name}: mean-to-train={mean_sim:.3f}  best-match={best_sim:.3f}")

print("-" * 40)
print(f"Real test-vs-train MEAN identity (ceiling): {np.mean(sims):.3f}")

20251006_001246_Instagram.jpg: mean-to-train=0.420  best-match=0.574
IMG_20260914_143012_478.jpg: mean-to-train=0.444  best-match=0.589
IMG_20260914_143051_167.jpg: mean-to-train=0.473  best-match=0.651
IMG_20260915_224144_012.jpg: mean-to-train=0.523  best-match=0.990
----------------------------------------
Real test-vs-train MEAN identity (ceiling): 0.465


Variance

In [19]:
import numpy as np
runs = {
  "baseline":       [-0.056,-0.006,-0.005,0.115,0.008,-0.006,0.053,0.007],
  "rank4_quality":  [0.206,0.349,0.233,0.322,0.245,0.250,0.247,0.269],
  "rank4_plain":    [0.206,0.051,0.259,0.103,0.255,0.054,0.254,0.139,0.224],
  "rank16_quality": [0.181,0.227,0.474,0.247,0.348,0.374,0.419,0.213,0.461,0.425],
}
for name, v in runs.items():
    v = np.array(v)
    print(f"{name:16s} mean={v.mean():.3f}  std={v.std():.3f}  n={len(v)}")

baseline         mean=0.014  std=0.047  n=8
rank4_quality    mean=0.265  std=0.044  n=8
rank4_plain      mean=0.172  std=0.081  n=9
rank16_quality   mean=0.337  std=0.105  n=10


Overfitting / language-drift check

In [20]:
import torch, gc
from diffusers import StableDiffusionPipeline
from src import config, evaluate

drift_dir = config.LORA_DIR.parent / "generated_drift"
drift_dir.mkdir(parents=True, exist_ok=True)
for p in drift_dir.glob("*.png"): p.unlink()

pipe = StableDiffusionPipeline.from_pretrained(config.BASE_MODEL)
pipe.load_lora_weights(str(config.LORA_DIR))      # rank-16 LoRA
pipe = pipe.to(config.DEVICE)
pipe.safety_checker = None

for k in range(10):
    g = torch.Generator(device=config.DEVICE).manual_seed(k)
    img = pipe("a photo of a man, color photograph, sharp focus, natural daylight",
               negative_prompt="sepia, grainy, vintage, black and white, blurry, low quality",
               num_inference_steps=40, guidance_scale=7.0, generator=g).images[0]
    img.save(drift_dir / f"drift_{k:03d}.png")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print("drift-test images saved")

evaluate.evaluate(generated_dir=drift_dir, train_dir=config.TRAIN_DIR)

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  7.87it/s]
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
 85%|████████▌ | 34/40 [2:12:22<23:21, 233.60s/it]  


KeyboardInterrupt: 

In [21]:
print(pipe.scheduler)

PNDMScheduler {
  "_class_name": "PNDMScheduler",
  "_diffusers_version": "0.41.0",
  "beta_end": 0.012,
  "beta_schedule": "scaled_linear",
  "beta_start": 0.00085,
  "clip_sample": false,
  "num_train_timesteps": 1000,
  "prediction_type": "epsilon",
  "set_alpha_to_one": false,
  "skip_prk_steps": true,
  "steps_offset": 1,
  "timestep_spacing": "leading",
  "trained_betas": null
}

